# 10 — Lag Features (May 2013 – Nov 2025)

## What This Notebook Does

This notebook takes the extended dataset (`remittance_2012_2025_extended.csv`) and adds **lag features** — columns that look back in time — so that machine learning models can learn from past migration patterns.

| Step | Action |
|------|--------|
| Load | Read extended CSV (160 rows, 5 columns) |
| Add lags | Create `dofe_lag3`, `dofe_lag6`, `dofe_lag9` columns |
| Drop rows | Remove first 9 rows where lag values are NaN |
| Save | Write model-ready CSV (151 rows, 8 columns) |

---

## Why Lag Features?

The extended CSV has `dofe_departures` — the number of Nepali workers who left for foreign employment **this month**. But remittance money does not arrive the same month a worker departs. A worker who leaves in August needs time to:
- Travel and settle in the destination country
- Start working and receive their first salary
- Send money back home

This delay is typically **3 to 9 months**. So we create three new columns:

| New Column | Meaning | Example for Nov 2025 |
|------------|---------|----------------------|
| `dofe_lag3` | Workers who left **3 months ago** | Aug 2025 departures |
| `dofe_lag6` | Workers who left **6 months ago** | May 2025 departures |
| `dofe_lag9` | Workers who left **9 months ago** | Feb 2025 departures |

These lagged values help the model understand: *"How many workers left some months ago, who are NOW sending money home this month?"*

---

## Why 9 Rows Are Dropped

The dataset starts at **Aug 2012**. When we shift the `dofe_departures` column back by 9 months, the first 9 rows cannot be filled — there is no data before Aug 2012 to look back into.

This is what the first 12 rows look like **before** dropping:

```
Row  Date      dofe_lag3   dofe_lag6   dofe_lag9
0    Aug 2012  NaN         NaN         NaN   ← missing all 3 lags
1    Sep 2012  NaN         NaN         NaN
2    Oct 2012  NaN         NaN         NaN
3    Nov 2012  45417       NaN         NaN   ← lag3 filled, others missing
4    Dec 2012  38297       NaN         NaN
5    Jan 2013  47067       NaN         NaN
6    Feb 2013  34990       45417       NaN   ← lag3+lag6 filled, lag9 missing
7    Mar 2013  54304       38297       NaN
8    Apr 2013  57951       47067       NaN
9    May 2013  51516       34990       45417 ← ALL 3 lags filled ✓
```

Rows 0–8 (Aug 2012 → Apr 2013) are dropped because `dofe_lag9` is NaN.

---

## What Would Happen If Rows Were NOT Dropped?

If you keep those 9 NaN rows and feed them into the models:

| Model | What happens |
|-------|--------------|
| **ARIMA** | Crashes or produces wrong results — cannot handle NaN in feature columns |
| **XGBoost** | May silently treat NaN as a special value — gives unreliable predictions for those rows |
| **LSTM** | Crashes immediately during training — neural networks cannot process NaN at all |

Dropping them is **not optional** — all three models require complete data to train correctly.

---

## Input / Output

```
remittance_2012_2025_extended.csv    ← READ ONLY — never changed (160 rows, 5 cols)
         |
         |  Step 10 adds lag columns
         |  Step 10 drops 9 NaN rows
         ↓
remittance_2012_2025_model_ready.csv ← NEW FILE created here (151 rows, 8 cols)
```

⚠️ **Important:** If you ever update `extended.csv` (add new months, fix values), you must re-run this notebook to regenerate `model_ready.csv`. It does **not** update automatically.

---

## Folder Structure Required

```
your_project/
├── notebook.ipynb
└── output/
    └── remittance_2012_2025_extended.csv
```

In [ ]:
import pandas as pd
import numpy as np

OUTPUT_FILE = 'output/remittance_2012_2025_extended.csv'
MODEL_READY = 'output/remittance_2012_2025_model_ready.csv'

## 1. Load Extended Data

In [ ]:
df = pd.read_csv(OUTPUT_FILE)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print('=== LOADED ===')
print(f'Rows    : {len(df)}')
print(f'Period  : {df["date"].iloc[0].strftime("%b %Y")} → '
      f'{df["date"].iloc[-1].strftime("%b %Y")}')
print(f'Columns : {df.columns.tolist()}')

## 2. Date Gap Check

If there is a gap in months, `shift()` will silently produce wrong lag values — so we check first.

In [ ]:
print('=== DATE GAP CHECK ===')
gaps = df['date'].diff().dropna()
bad  = gaps[gaps > pd.Timedelta(days=35)]
if len(bad) == 0:
    print('✓ No gaps — safe to apply shift()')
else:
    print(f'⚠ {len(bad)} gap(s) found — fix before continuing')
    for idx in bad.index:
        print(f'  {df.loc[idx-1,"date"].strftime("%b %Y")} → '
              f'{df.loc[idx,"date"].strftime("%b %Y")}')

## 3. Add Lag Features

`shift(n)` moves the column **down by n rows**, so each row sees the value from n months earlier.

- `dofe_lag3` → workers who left 3 months ago
- `dofe_lag6` → workers who left 6 months ago  
- `dofe_lag9` → workers who left 9 months ago

The first 9 rows will show NaN — this is expected and handled in the next step.

In [ ]:
df['dofe_lag3'] = df['dofe_departures'].shift(3)
df['dofe_lag6'] = df['dofe_departures'].shift(6)
df['dofe_lag9'] = df['dofe_departures'].shift(9)

print('=== FIRST 12 ROWS AFTER SHIFT ===')
print('(First 9 rows show NaN — expected)')
print(df[['date','dofe_departures',
          'dofe_lag3','dofe_lag6','dofe_lag9']].head(12).to_string())

## 4. Drop First 9 Rows

Rows 0–8 (Aug 2012 → Apr 2013) have NaN in `dofe_lag9` because the dataset starts at Aug 2012 and there is no earlier data to look 9 months back into.

These rows **must** be removed — ARIMA, XGBoost, and LSTM all crash or produce wrong results when trained on NaN values.

In [ ]:
df_model = df.dropna(subset=['dofe_lag9']).reset_index(drop=True)

print('=== ROWS AFTER DROPPING ===')
print(f'Before : {len(df)} rows  Aug 2012 → Nov 2025')
print(f'After  : {len(df_model)} rows  '
      f'{df_model["date"].iloc[0].strftime("%b %Y")} → '
      f'{df_model["date"].iloc[-1].strftime("%b %Y")}')
print(f'Dropped: 9 rows (Aug 2012 → Apr 2013) — dofe_lag9 was NaN')

## 5. Verify Lag Values

Cross-check that each lag column contains the correct historical value by comparing against the original extended data.

In [ ]:
print('=== VERIFICATION: First row (May 2013) ===')
first = df_model.iloc[0]
for months, col in [(3,'dofe_lag3'),(6,'dofe_lag6'),(9,'dofe_lag9')]:
    target   = first['date'] - pd.DateOffset(months=months)
    expected = df[df['date'] == target]['dofe_departures'].values[0]
    actual   = first[col]
    status   = '✓' if actual == expected else '✗'
    print(f'  {col} → {target.strftime("%b %Y")} '
          f'expected {expected:.0f} got {actual:.0f} {status}')

print()
print('=== VERIFICATION: Last row (Nov 2025) ===')
last = df_model.iloc[-1]
for months, col in [(3,'dofe_lag3'),(6,'dofe_lag6'),(9,'dofe_lag9')]:
    target   = last['date'] - pd.DateOffset(months=months)
    expected = df[df['date'] == target]['dofe_departures'].values[0]
    actual   = last[col]
    status   = '✓' if actual == expected else '✗'
    print(f'  {col} → {target.strftime("%b %Y")} '
          f'expected {expected:.0f} got {actual:.0f} {status}')

## 6. Final Checks

In [ ]:
print('=== MISSING VALUES (all must be 0) ===')
print(df_model.isnull().sum())

print()
print('=== SUMMARY STATISTICS ===')
print(df_model[['remittance','exchange_rate','oil_price',
                'dofe_departures','dofe_lag3',
                'dofe_lag6','dofe_lag9']].describe().round(2))

print()
print('=== LAST 5 ROWS ===')
print(df_model.tail().to_string())

## 7. Save

In [ ]:
df_model.to_csv(MODEL_READY, index=False)
print(f'✅ Saved → {MODEL_READY}')
print(f'   Shape   : {df_model.shape}')
print(f'   Columns : {df_model.columns.tolist()}')
print()
print('   Column guide:')
print('   date            → month')
print('   remittance      → TARGET (what the model predicts)')
print('   exchange_rate   → feature')
print('   oil_price       → feature')
print('   dofe_departures → workers who left THIS month')
print('   dofe_lag3       → workers who left 3 months ago')
print('   dofe_lag6       → workers who left 6 months ago')
print('   dofe_lag9       → workers who left 9 months ago')
print()
print('   ⚠ If you update extended.csv, re-run this notebook!')